In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MHA(nn.Module):

    def __init__(self, input_dim, num_head):
        super().__init__()
        assert input_dim % num_head == 0
        self.input_dim = input_dim
        self.num_head = num_head
        self.d_k = input_dim // num_head
        self.wqkv = nn.Linear(input_dim, input_dim * 3)
        self.wo = nn.Linear(input_dim, input_dim)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        qkv = self.wqkv(x)
        q, k, v = torch.chunk(qkv, 3, dim=-1)
        q = q.view(B, L, self.num_head, self.d_k).transpose(1, 2)
        k = k.view(B, L, self.num_head, self.d_k).transpose(1, 2)
        v = v.view(B, L, self.num_head, self.d_k).transpose(1, 2)
        attention = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.d_k)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e9)
        attention_scores = F.softmax(attention, dim=-1)
        context = torch.matmul(attention_scores, v)
        context = context.transpose(1, 2).contiguous().view(B, L, D)
        output = self.wo(context)
        return output
B, L, D = 10, 20, 36
x = torch.randn(B, L, D)
mha = MHA(D, 2)

print(mha(x).shape)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MQA(nn.Module):

    def __init__(self, input_dim, num_head):
        super().__init__()
        assert input_dim % num_head == 0
        self.input_dim = input_dim
        self.num_head = num_head
        self.d_k = input_dim // num_head
        self.wq = nn.Linear(input_dim, input_dim)
        self.wkv = nn.Linear(input_dim, self.d_k * 2)
        self.wo = nn.Linear(input_dim, input_dim)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        q = self.wq(x)
        kv = self.wkv(x)
        k, v = torch.chunk(kv, 2, dim=-1)
        q = q.view(B, L, self.num_head, self.d_k).transpose(1, 2)
        k = torch.unsqueeze(k, 1)   # (B, 1, L, d_k)
        v = torch.unsqueeze(v, 1)   # (B, 1, L, d_k)
        attention = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.d_k)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e9)
        attention_scores = F.softmax(attention, dim=-1)
        context = torch.matmul(attention_scores, v)
        context = context.transpose(1, 2).contiguous().view(B, L, D)
        output = self.wo(context)
        return output
B, L, D = 10, 20, 36
x = torch.randn(B, L, D)
mha = MQA(D, 2)

print(mha(x).shape)

torch.Size([10, 20, 36])


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class GQA(nn.Module):

    def __init__(self, input_dim, q_num_head, kv_num_head):
        super().__init__()
        assert input_dim % q_num_head == 0
        assert q_num_head % kv_num_head == 0
        self.input_dim = input_dim
        self.q_num_head = q_num_head
        self.kv_num_head = kv_num_head
        self.d_k = input_dim // q_num_head
        self.num_q_group = q_num_head // kv_num_head

        self.wq = nn.Linear(input_dim, input_dim)
        self.wkv = nn.Linear(input_dim, self.d_k * kv_num_head * 2)
        self.wo = nn.Linear(input_dim, input_dim)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        q = self.wq(x)
        q = q.view(B, L, self.q_num_head, self.d_k).transpose(1, 2).contiguous().view(B, self.kv_num_head, self.num_q_group, L, self.d_k)
        kv = self.wkv(x)
        k, v = torch.chunk(kv, 2, dim=-1)
        k = k.view(B, L, self.kv_num_head, self.d_k).transpose(1, 2)
        v = v.view(B, L, self.kv_num_head, self.d_k).transpose(1, 2)
        k = torch.unsqueeze(k, 2)   # (B, self.kv_num_head, 1, L, d_k)
        v = torch.unsqueeze(v, 2)   # (B, self.kv_num_head, 1, L, d_k)
        attention = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.d_k)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e9)
        attention_scores = F.softmax(attention, dim=-1)
        context = torch.matmul(attention_scores, v)
        context = context.view(B, self.q_num_head, L, self.d_k).transpose(1, 2).contiguous().view(B, L, D)
        output = self.wo(context)
        return output
B, L, D = 10, 20, 36
x = torch.randn(B, L, D)
mha = GQA(D, 4, 2)

print(mha(x).shape)

torch.Size([10, 20, 36])


In [14]:
import numpy as np
def auc_rank(q_list, label):
    q_list = np.array(q_list)
    label = np.array(label)
    rank_index = np.argsort(q_list)
    q_list_ranked = q_list[rank_index]
    label_ranked = label[rank_index]
    total_pos = np.sum(label_ranked == 1)
    total_neg = np.sum(label_ranked == 0)
    l, n= 0, len(label_ranked)
    cum_neg, cum_pos = 0, 0
    while l < n:
        r = l
        while r < n and q_list_ranked[l] == q_list_ranked[r]:
            r += 1
        group_neg = np.sum(label_ranked[l:r] == 0)
        group_pos = np.sum(label_ranked[l:r] == 1)
        cum_pos  += group_pos * cum_neg + group_pos * group_neg * 0.5
        cum_neg += group_neg
        l = r
    return cum_pos / (total_pos * total_neg)


q = [0.1, 0.9, 0.2, 0.8, 1, 0.2, 0.3, 0.8]
label = [0, 1, 0, 0, 1, 0, 0, 1]
auc  = auc_rank(q, label)
print(f"auc:{auc}")

auc:0.9666666666666667


In [15]:
import numpy as np
from sklearn.metrics import roc_auc_score

def auc_rank_fixed(q_list, label):
    q_list = np.array(q_list, dtype=float)
    label  = np.array(label)

    # ── Step 1：按分数从小到大排序 ──────────────────────────────────────
    # argsort 返回的是"排序后的原始下标"，例如：
    #   q = [0.1, 0.9, 0.2, 0.8, 1.0, 0.2, 0.3, 0.8]
    #   rank_index = [0, 2, 5, 6, 3, 7, 1, 4]  （分数从小到大对应的原始位置）
    rank_index   = np.argsort(q_list)
    q_sorted     = q_list[rank_index]    # 分数从小到大
    label_sorted = label[rank_index]     # label 跟着分数一起排好序

    total_pos = np.sum(label == 1)       # 正样本总数
    total_neg = np.sum(label == 0)       # 负样本总数

    # ── Step 2：理解 AUC 的含义 ─────────────────────────────────────────
    # AUC = "随机取一个正样本和一个负样本，正样本分数 > 负样本分数" 的概率
    # 具体：
    #   concordant（正确序对）：正样本分数 > 负样本分数，贡献 1.0
    #   tie（平局）：           正样本分数 = 负样本分数，贡献 0.5
    #   discordant（错误序对）：正样本分数 < 负样本分数，贡献 0.0
    # AUC = 所有正负样本对的贡献之和 / 正负样本对总数

    cum_neg    = 0     # 记录"当前组之前"已经遍历过的负样本数量
    concordant = 0.0   # 累计贡献
    i, n       = 0, len(label_sorted)

    # ── Step 3：分组遍历（每组内分数完全相同）───────────────────────────
    while i < n:

        # 3a. 找到与位置 i 分数相同的区间 [i, j)
        #     例如 score=0.2 出现在位置 2 和 3，则 i=2, j=4
        j = i
        while j < n and q_sorted[j] == q_sorted[i]:
            j += 1
        # 现在 label_sorted[i:j] 就是当前分数组内的所有样本

        group_pos = np.sum(label_sorted[i:j] == 1)  # 组内正样本数
        group_neg = np.sum(label_sorted[i:j] == 0)  # 组内负样本数

        # 3b. 计算当前组内正样本与"所有负样本"形成的序对贡献
        #
        #   情况①：组前负样本（cum_neg 个）
        #     这些负样本分数 < 当前组分数
        #     → 每个正样本都比它们分数高 → 每对贡献 1.0
        #     → 新增 concordant = group_pos × cum_neg × 1.0
        #
        #   情况②：组内负样本（group_neg 个）
        #     这些负样本与当前组正样本分数相同 → tie
        #     → 每对贡献 0.5
        #     → 新增 concordant = group_pos × group_neg × 0.5
        concordant += group_pos * cum_neg + group_pos * group_neg * 0.5

        # 3c. 当前组遍历完毕，把组内负样本数累加到 cum_neg
        #     供后续更高分数的正样本使用
        cum_neg += group_neg
        i = j   # 移动到下一组

    # ── Step 4：归一化 ──────────────────────────────────────────────────
    # 正负样本对总数 = total_pos × total_neg
    return concordant / (total_pos * total_neg)


# ── 测试 ────────────────────────────────────────────────────────────────
label = [0, 1, 0, 0, 1, 0, 0, 1]
q     = [0.1, 0.9, 0.2, 0.8, 1, 0.2, 0.3, 0.8]

print(f"修复版  AUC: {auc_rank_fixed(q, label)}")
print(f"sklearn AUC: {roc_auc_score(label, q)}")

修复版  AUC: 0.9666666666666667
sklearn AUC: 0.9666666666666667


In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, activation="gelu", norm="LN", dropout=0.1):        
        super().__init__()
        layer_dim = [input_dim] + hidden_dim + [output_dim]
        layers = []                   
        n = len(layer_dim)

        for i in range(n - 1):
            layers.append(nn.Linear(layer_dim[i], layer_dim[i + 1]))

            if i < n - 2:
                if norm == "LN":                                    
                    layers.append(nn.LayerNorm(layer_dim[i + 1]))  
                elif norm == "BN":                                 
                    layers.append(nn.BatchNorm1d(layer_dim[i + 1]))
                else:
                    raise ValueError(f"不支持的归一化: {norm}")

                if activation == "relu":
                    layers.append(nn.ReLU())
                elif activation == "tanh":
                    layers.append(nn.Tanh())
                elif activation == "sigmoid":
                    layers.append(nn.Sigmoid())
                elif activation == "gelu":                          
                    layers.append(nn.GELU())
                else:
                    raise ValueError(f"不支持的激活函数: {activation}")
                if dropout > 0:                                    
                    layers.append(nn.Dropout(dropout))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# ===== 验证 =====
model = MLP(input_dim=64, hidden_dim=[128, 128], output_dim=10, norm="LN")
x = torch.randn(32, 64)
print(model(x).shape)   

torch.Size([32, 10])


In [32]:
import torch
def cross_entropy(logits, target):
    B, D = logits.shape
    logit_max = torch.max(logits, dim=-1)[0]
    logit_stable = logits - logit_max
    logit_log_sum_exp = torch.log(torch.sum(torch.exp(logit_stable), dim=-1))
    pos_logits = logit_stable[torch.arange(B), target]
    loss = - pos_logits + logit_log_sum_exp
    return loss.mean()
logits = torch.tensor([[1.0000, 0.4985, 0.6664, 0.2533],
                    [0.4985, 1.0000, 0.8408, 0.5431],
                    [0.6664, 0.8408, 1.0000, 0.8372],
                    [0.2533, 0.5431, 0.8372, 1.0000]])
target = torch.tensor([0, 1, 2, 3])
print(cross_entropy(logits, target))

tensor(1.1176)


In [ ]:
import torch
import torch.nn.functional as F
def BCE_loss(logits, label):
    loss = torch.clamp(logits, min=0) - logits * label + torch.log(1 + torch.exp(-torch.abs(logits)))
    return loss.mean()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
def infoNce(user_emb, item_emb, temperature=1.0):
    """
    user_emb: (B, D)
    item_emb: (B, D)
    """
    user_emb = F.normalize(user_emb, dim=-1)
    item_emb = F.normalize(item_emb, dim=-1)
    sim = torch.matmul(user_emb, item_emb.transpose(0, 1)) / temperature
    label = torch.arange(user_emb.size(0))
    loss = F.cross_entropy(sim, label)
    return loss.mean()

In [1]:
import numpy as np
def auc_rank(q, label):
    q = np.array(q)
    label = np.array(label)
    rank_index = np.argsort(q)
    q_rank = q[rank_index]
    label_rank = label[rank_index]
    t_pos = np.sum(label == 1)
    t_neg = np.sum(label == 0)
    l, n = 0, len(q)
    cum_neg, cum_pos = 0, 0
    while l < n:
        r = l
        while r < n and q_rank[l] == q_rank[r]:
            r += 1
        group_neg = np.sum(label_rank[l:r] == 0)
        group_pos = np.sum(label_rank[l:r] == 1)
        cum_pos += group_pos * cum_neg + group_pos * group_neg * 0.5
        cum_neg += group_neg
        l = r
    return cum_pos / (t_pos*t_neg)
q = [0.1, 0.9, 0.2, 0.8, 1, 0.2, 0.3, 0.8]
label = [0, 1, 0, 0, 1, 0, 0, 1]
auc  = auc_rank(q, label)
print(f"auc:{auc}")

auc:0.9666666666666667


In [ ]:
class TreeNode:
    def __init(self, val, left, right):
        self.val = val
        self.left = left
        self.right = right
class Solution:
    def __init__(self):
        # 创建一个链表作为结果容器
        self.res = []

    # 返回前序遍历结果
    def preorder(self, root: TreeNode):
        self.traverse(root)
        return self.res

    # 二叉树遍历函数
    def traverse(self, root: TreeNode):
        if not root:
            return
        # 前序遍历位置
        self.res.append(root.val)
        self.traverse(root.left)
        self.traverse(root.right)